# RuDevices Benchmark — транскрибация

Запускается на GPU-машине. Сохраняет только гипотезы:
```
asr_data/rudevices/   {model}_{timestamp}_results.jsonl
```
Каждая строка: `{audio_path, reference, hypothesis, time}`

Датасет: [SOVA RuDevices](https://github.com/sovaai/sova-dataset) — ~10 000 сэмплов живой речи, записанной на мобильных устройствах.

In [ ]:
import subprocess, sys, importlib

def _ensure_pip():
    """Bootstrap pip if missing (happens in some venvs without pip)."""
    try:
        import pip  # noqa: F401
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "-q"])

def _pip_install(*packages):
    _ensure_pip()
    subprocess.check_call([sys.executable, "-m", "pip", "install", *packages, "-q"])

# huggingface_hub должен быть достаточно свежим для transformers
_pip_install("huggingface_hub>=0.23", "--upgrade")

# Ставим всё нужное для бенчмарка одной командой
_pip_install(
    "git+https://github.com/akatsnelson/plantain2asr.git"
    "#egg=plantain2asr[gigaam,whisper,vosk]",
    "--upgrade",
)

# tone устанавливается отдельно (git-only пакет)
try:
    import tone  # noqa: F401
except ImportError:
    _pip_install("git+https://github.com/voicekit-team/T-one.git")
    importlib.invalidate_caches()

# miniaudio нужен для read_audio в T-one
try:
    import miniaudio  # noqa: F401
except ImportError:
    _pip_install("miniaudio")
    importlib.invalidate_caches()

# onnxruntime-gpu вместо CPU-only onnxruntime (нужен для T-one на CUDA)
import onnxruntime as _ort
if "CUDAExecutionProvider" not in _ort.get_available_providers():
    _pip_install("onnxruntime-gpu", "--upgrade")
    importlib.invalidate_caches()
    print("⚠️  onnxruntime-gpu установлен — нужен рестарт ядра")

print("✅ Все зависимости установлены")

In [ ]:
import os, json, glob, warnings
from datetime import datetime
warnings.filterwarnings('ignore')

from plantain2asr import RuDevicesDataset, Models

RUDEVICES_DIR = "data/RuDevices"
OUT_DIR = "asr_data/rudevices"
os.makedirs(OUT_DIR, exist_ok=True)

ds = RuDevicesDataset(RUDEVICES_DIR)
print(f"Всего: {len(ds)} семплов")

In [ ]:
def save(dataset, model_name, out_dir):
    ts   = datetime.now().strftime("%Y%m%d_%H%M%S")
    path = os.path.join(out_dir, f"{model_name}_{ts}_results.jsonl")
    n = 0
    with open(path, "w", encoding="utf-8") as f:
        for s in dataset:
            res = s.asr_results.get(model_name)
            if res is None:
                continue
            f.write(json.dumps({
                "audio_path" : s.audio_path,
                "reference"  : s.text or "",
                "hypothesis" : res.get("hypothesis") or "",
                "time"       : res.get("processing_time") or res.get("time") or 0.0,
            }, ensure_ascii=False) + "\n")
            n += 1
    print(f"  → {path}  ({n} строк)")

## Инференс

In [ ]:
# ── GigaAM v3 e2e-RNNT ─────────────────────────────────────────────────
model = Models.GigaAM_v3(model_name="e2e_rnnt")
ds >> model
save(ds, model.name, OUT_DIR)
del model

In [ ]:
# ── GigaAM v3 e2e-CTC ──────────────────────────────────────────────────
model = Models.GigaAM_v3(model_name="e2e_ctc")
ds >> model
save(ds, model.name, OUT_DIR)
del model

In [ ]:
# ── GigaAM v3 RNNT ─────────────────────────────────────────────────────
model = Models.GigaAM_v3(model_name="rnnt")
ds >> model
save(ds, model.name, OUT_DIR)
del model

In [ ]:
# ── GigaAM v2 RNNT ─────────────────────────────────────────────────────
model = Models.GigaAM_v2(model_name="v2_rnnt")
ds >> model
save(ds, model.name, OUT_DIR)
del model

In [ ]:
# ── Whisper ─────────────────────────────────────────────────────────────
model = Models.Whisper(model_name="bond005/whisper-large-v3-ru-podlodka")
ds >> model
save(ds, model.name, OUT_DIR)
del model

In [ ]:
# ── T-one RussianTone ───────────────────────────────────────────────────
model = Models.Tone()
ds >> model
save(ds, model.name, OUT_DIR)
del model

In [ ]:
# ── NVIDIA Canary ───────────────────────────────────────────────────────
model = Models.Canary()
ds >> model
save(ds, model.name, OUT_DIR)
del model

In [ ]:
# ── Vosk (offline, CPU) ─────────────────────────────────────────────────
model = Models.Vosk(model_path="models/vosk-model-ru-0.42")
ds >> model
save(ds, model.name, OUT_DIR)
del model

In [ ]:
# ── SaluteSpeech API ────────────────────────────────────────────────────
if os.getenv("SALUTE_AUTH_DATA"):
    model = Models.SaluteSpeech()
    ds >> model
    save(ds, model.name, OUT_DIR)
    del model
else:
    print("⚠️  SALUTE_AUTH_DATA не задан — пропускаем")

In [ ]:
# ── Итог ────────────────────────────────────────────────────────────────
files = sorted(glob.glob(os.path.join(OUT_DIR, "*_results.jsonl")))
print(f"{OUT_DIR}/  ({len(files)} файлов)")
for p in files:
    n = sum(1 for _ in open(p))
    print(f"  {os.path.basename(p):<55} {n} строк")